# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I use a Random Forest Regressor because the lane is a ranking problem with mixed performance signals and potentially nonlinear relationships. The model is used to produce a continuous priority score that can be ranked, rather than as an automatic refresh decision. A tree-based model also gives a simple feature-importance view for error analysis.

In [22]:
from getpass import getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

jan_sample = con.sql(
    f"SELECT * FROM {PERF_JAN}"
).df()

jan_sample["report_date"] = pd.to_datetime(
    jan_sample["report_date"]
)

print("Shape:", jan_sample.shape)
print("Columns:")
print(jan_sample.columns.tolist())

Shape: (1297, 31)
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [23]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print(model)

RandomForestRegressor(max_depth=6, min_samples_leaf=5, n_estimators=300,
                      n_jobs=-1, random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a time-aware split because the practical question is whether earlier observations can help rank content for a later review period. Training uses January 27–29 and evaluation uses January 30–31. I avoid a random split because the same content can appear on multiple dates, which could make a random split overly optimistic.

In [24]:
data = jan_sample.copy()

data["report_date"] = pd.to_datetime(
    data["report_date"]
)

train_end = pd.Timestamp("2025-01-29")

train_dates = data[
    data["report_date"] <= train_end
]

test_dates = data[
    data["report_date"] > train_end
]

print("All dates:")
print(sorted(data["report_date"].unique()))

print("\nTrain rows:", len(train_dates))
print("Test rows:", len(test_dates))

print("\nTrain dates:")
print(
    train_dates["report_date"]
    .value_counts()
    .sort_index()
)

print("\nTest dates:")
print(
    test_dates["report_date"]
    .value_counts()
    .sort_index()
)

All dates:
[Timestamp('2025-01-27 00:00:00'), Timestamp('2025-01-28 00:00:00'), Timestamp('2025-01-29 00:00:00'), Timestamp('2025-01-30 00:00:00'), Timestamp('2025-01-31 00:00:00')]

Train rows: 882
Test rows: 415

Train dates:
report_date
2025-01-27    303
2025-01-28    317
2025-01-29    262
Name: count, dtype: int64

Test dates:
report_date
2025-01-30    194
2025-01-31    221
Name: count, dtype: int64


In [18]:
# Sort by content and time.
data = data.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).copy()

data["next_date"] = (
    data.groupby(["client_hash_id", "content_hash_id"])["report_date"]
    .shift(-1)
)

data["days_to_next"] = (
    data["next_date"] - data["report_date"]
).dt.days

print(data["days_to_next"].value_counts(dropna=False).sort_index())

print(data[
    [
        "report_date",
        "content_hash_id",
        "gsc_clicks",
        "next_day_clicks"
    ]
].head(10))

days_to_next
1.0    780
2.0     34
3.0      5
4.0      2
NaN    476
Name: count, dtype: int64
     report_date           content_hash_id  gsc_clicks  next_day_clicks
79    2025-01-27  content_0217be03126aa7a5           0              0.0
381   2025-01-28  content_0217be03126aa7a5           0              0.0
697   2025-01-29  content_0217be03126aa7a5           0              0.0
957   2025-01-30  content_0217be03126aa7a5           0              0.0
1154  2025-01-31  content_0217be03126aa7a5           0              NaN
63    2025-01-27  content_0642dc7f62d4f780           1              1.0
366   2025-01-28  content_0642dc7f62d4f780           1              0.0
681   2025-01-29  content_0642dc7f62d4f780           0              0.0
941   2025-01-30  content_0642dc7f62d4f780           0              2.0
1138  2025-01-31  content_0642dc7f62d4f780           2              NaN


In [25]:
data = data.sort_values(
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
).copy()

group_cols = [
    "client_hash_id",
    "content_hash_id"
]

data["next_date"] = (
    data.groupby(group_cols)["report_date"]
    .shift(-1)
)

data["days_to_next"] = (
    data["next_date"] - data["report_date"]
).dt.days

print("Days to next observation:")
print(
    data["days_to_next"]
    .value_counts(dropna=False)
    .sort_index()
)

Days to next observation:
days_to_next
1.0    780
2.0     34
3.0      5
4.0      2
NaN    476
Name: count, dtype: int64


In [26]:
data["next_day_clicks"] = (
    data.groupby(group_cols)["gsc_clicks"]
    .shift(-1)
)

# Keep the target only when the next observation
# is exactly one calendar day later.
data.loc[
    data["days_to_next"] != 1,
    "next_day_clicks"
] = np.nan

model_data = data.dropna(
    subset=["next_day_clicks"]
).copy()

print(
    "Rows with valid next-day target:",
    len(model_data)
)

Rows with valid next-day target: 780


In [11]:
import numpy as np
model_data["ctr"] = np.where(
    model_data["gsc_impressions"] > 0,
    model_data["gsc_clicks"] / model_data["gsc_impressions"],
    np.nan
)

model_data["total_sessions"] = (
    model_data["sessions_organic"].fillna(0)
    + model_data["sessions_direct"].fillna(0)
    + model_data["sessions_referral"].fillna(0)
    + model_data["sessions_social"].fillna(0)
    + model_data["sessions_paid"].fillna(0)
    + model_data["sessions_ai"].fillna(0)
)

model_data["organic_share"] = np.where(
    model_data["total_sessions"] > 0,
    model_data["sessions_organic"]
    / model_data["total_sessions"],
    np.nan
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
]

X = model_data[features].copy()
y = model_data["next_day_clicks"].copy()

# Tree models don't accept NaN.
X = X.fillna(0)

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']
X shape: (821, 14)
y shape: (821,)


In [27]:
model_data["ctr"] = np.where(
    model_data["gsc_impressions"] > 0,
    model_data["gsc_clicks"]
    / model_data["gsc_impressions"],
    np.nan
)

model_data["total_sessions"] = (
    model_data["sessions_organic"].fillna(0)
    + model_data["sessions_direct"].fillna(0)
    + model_data["sessions_referral"].fillna(0)
    + model_data["sessions_social"].fillna(0)
    + model_data["sessions_paid"].fillna(0)
    + model_data["sessions_ai"].fillna(0)
)

model_data["organic_share"] = np.where(
    model_data["total_sessions"] > 0,
    model_data["sessions_organic"]
    / model_data["total_sessions"],
    np.nan
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
]

X = model_data[features].copy()
y = model_data["next_day_clicks"].copy()

# Random Forest cannot accept NaN.
X = X.fillna(0)

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 14
X shape: (780, 14)
y shape: (780,)


In [28]:
train_mask = (
    model_data["report_date"]
    <= train_end
)

test_mask = (
    model_data["report_date"]
    > train_end
)

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print("Training rows:", len(X_train))
print("Evaluation rows:", len(X_test))

print("\nTraining date range:")
print(
    model_data.loc[
        train_mask,
        "report_date"
    ].min(),
    "to",
    model_data.loc[
        train_mask,
        "report_date"
    ].max()
)

print("\nEvaluation date range:")
print(
    model_data.loc[
        test_mask,
        "report_date"
    ].min(),
    "to",
    model_data.loc[
        test_mask,
        "report_date"
    ].max()
)

Training rows: 600
Evaluation rows: 180

Training date range:
2025-01-27 00:00:00 to 2025-01-29 00:00:00

Evaluation date range:
2025-01-30 00:00:00 to 2025-01-30 00:00:00


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare the Random Forest with the ML-07-style rule baseline on the same evaluation observations. The observed outcome is next-day clicks because the dataset does not contain human labels for whether a page actually needed a refresh. NDCG@10 measures whether higher-click outcomes are concentrated near the top of the ranked queue.

In [29]:
model.fit(
    X_train,
    y_train
)

model_pred = model.predict(
    X_test
)

print("Model trained.")
print("Predictions:", len(model_pred))

Model trained.
Predictions: 180


In [ ]:
# Restore the exact row order used by X_test/model_pred.
eval_data = model_data.loc[test_mask].copy()

# Recalculate baseline columns on this original order.

# A simple baseline using observed current-day signals.
eval_data["baseline_score"] = 0

ctr_threshold = model_data.loc[
    train_mask, "ctr"
].quantile(0.25)

position_threshold = model_data.loc[
    train_mask, "gsc_avg_position"
].quantile(0.75)

eval_data.loc[
    eval_data["ctr"] <= ctr_threshold,
    "baseline_score"
] += 2

eval_data.loc[
    eval_data["gsc_avg_position"] >= position_threshold,
    "baseline_score"
] += 2

print(eval_data[
    [
        "content_hash_id",
        "baseline_score",
        "next_day_clicks"
    ]
].head())

              content_hash_id  baseline_score  next_day_clicks
957  content_0217be03126aa7a5               2              0.0
941  content_0642dc7f62d4f780               2              2.0
965  content_0672d8db776419c0               2              0.0
931  content_0ca502c18c4fd41e               2              0.0
982  content_0d308caf94a3ed16               2              0.0


In [30]:
train_data = model_data.loc[
    train_mask
].copy()

eval_data = model_data.loc[
    test_mask
].copy()

# -----------------------------
# Thresholds from training data
# -----------------------------

ctr_threshold = train_data[
    "ctr"
].quantile(0.25)

position_threshold = train_data[
    "gsc_avg_position"
].quantile(0.75)

if train_data["organic_share"].notna().any():
    organic_share_threshold = train_data[
        "organic_share"
    ].quantile(0.25)
else:
    organic_share_threshold = np.nan

print("CTR threshold:", ctr_threshold)
print("Position threshold:", position_threshold)
print(
    "Organic-share threshold:",
    organic_share_threshold
)

CTR threshold: 0.0
Position threshold: 48.962500000000006
Organic-share threshold: nan


In [31]:
history = data.sort_values(
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
).copy()

history["previous_day_clicks"] = (
    history.groupby(group_cols)["gsc_clicks"]
    .shift(1)
)

previous_clicks = history[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "previous_day_clicks"
    ]
].copy()

eval_data = eval_data.merge(
    previous_clicks,
    on=[
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ],
    how="left",
    sort=False
)

In [32]:
eval_data["baseline_score"] = 0

# -----------------------------
# TREND_DOWN = 3 points
# -----------------------------

trend_down = (
    eval_data["previous_day_clicks"].notna()
    & (
        eval_data["gsc_clicks"]
        < eval_data["previous_day_clicks"]
    )
)

eval_data.loc[
    trend_down,
    "baseline_score"
] += 3

# -----------------------------
# LOW_CTR = 2 points
# -----------------------------

eval_data.loc[
    eval_data["ctr"] <= ctr_threshold,
    "baseline_score"
] += 2

# -----------------------------
# WEAK_POSITION = 2 points
# -----------------------------

eval_data.loc[
    eval_data["gsc_avg_position"]
    >= position_threshold,
    "baseline_score"
] += 2

# -----------------------------
# LOW_ORGANIC_SHARE = 1 point
# -----------------------------

if not np.isnan(organic_share_threshold):
    eval_data.loc[
        eval_data["organic_share"]
        <= organic_share_threshold,
        "baseline_score"
    ] += 1

print("Baseline score distribution:")
print(
    eval_data["baseline_score"]
    .value_counts()
    .sort_index()
)

display(
    eval_data[
        [
            "content_hash_id",
            "report_date",
            "baseline_score",
            "next_day_clicks"
        ]
    ].head(10)
)

Baseline score distribution:
baseline_score
0     16
2    105
4     47
5     12
Name: count, dtype: int64


,content_hash_id,report_date,baseline_score,next_day_clicks
0,content_0217be03126aa7a5,2025-01-30,2,0.0
1,content_0642dc7f62d4f780,2025-01-30,2,2.0
2,content_0672d8db776419c0,2025-01-30,2,0.0
3,content_0ca502c18c4fd41e,2025-01-30,2,0.0
4,content_0d308caf94a3ed16,2025-01-30,2,0.0
5,content_0fa49fb87f830edf,2025-01-30,2,0.0
6,content_132cfd61ee6071be,2025-01-30,5,2.0
7,content_1513a75b3809e368,2025-01-30,2,0.0
8,content_167aff5fed6593c7,2025-01-30,2,0.0
9,content_17c76438024dece9,2025-01-30,2,0.0


In [33]:
prediction_data = model_data.loc[
    test_mask,
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
].copy()

prediction_data["model_score"] = model_pred

eval_data = eval_data.merge(
    prediction_data,
    on=[
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ],
    how="left",
    sort=False
)

print(
    "Missing model predictions:",
    eval_data["model_score"].isna().sum()
)

print(
    "Evaluation rows:",
    len(eval_data)
)

Missing model predictions: 0
Evaluation rows: 180


In [34]:
from sklearn.metrics import ndcg_score

daily_results = []

for date, day_df in eval_data.groupby(
    "report_date",
    sort=True
):

    y_true = day_df[
        "next_day_clicks"
    ].to_numpy()

    baseline_scores = day_df[
        "baseline_score"
    ].to_numpy()

    model_scores = day_df[
        "model_score"
    ].to_numpy()

    # If every item has zero next-day clicks,
    # NDCG is not informative for that day.
    if y_true.max() == 0:
        print(
            f"{date.date()}: "
            "no positive next-day clicks; "
            "NDCG skipped."
        )
        continue

    k = min(
        10,
        len(day_df)
    )

    baseline_ndcg = ndcg_score(
        [y_true],
        [baseline_scores],
        k=k
    )

    model_ndcg = ndcg_score(
        [y_true],
        [model_scores],
        k=k
    )

    daily_results.append({
        "report_date": date,
        "Rule baseline NDCG@10": baseline_ndcg,
        "Random Forest NDCG@10": model_ndcg
    })

daily_ndcg = pd.DataFrame(
    daily_results
)

display(daily_ndcg)

,report_date,Rule baseline NDCG@10,Random Forest NDCG@10
0,2025-01-30,0.066051,0.689987


In [35]:
comparison = pd.DataFrame({
    "method": [
        "Rule baseline",
        "Random Forest"
    ],
    "NDCG@10": [
        daily_ndcg[
            "Rule baseline NDCG@10"
        ].mean(),
        daily_ndcg[
            "Random Forest NDCG@10"
        ].mean()
    ]
})

display(comparison)

,method,NDCG@10
0,Rule baseline,0.066051
1,Random Forest,0.689987


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The largest errors are useful for checking where the model's predicted priority differs from the observed next-day clicks. Feature importance is used only as a directional interpretation of what the Random Forest relied on in this run.

In [36]:
eval_results = eval_data[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "next_day_clicks",
        "baseline_score",
        "model_score"
    ]
].copy()

eval_results["absolute_error"] = (
    eval_results["next_day_clicks"]
    - eval_results["model_score"]
).abs()

display(
    eval_results.sort_values(
        "absolute_error",
        ascending=False
    ).head(10)
)

,client_hash_id,content_hash_id,report_date,next_day_clicks,baseline_score,model_score,absolute_error
139,client_9958f0a7ae1df715,content_f94fe855380e150f,2025-01-30,5.0,0,2.135390,2.864610
6,client_9958f0a7ae1df715,content_132cfd61ee6071be,2025-01-30,2.0,5,0.001701,1.998299
21,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,2025-01-30,4.0,0,2.031996,1.968004
1,client_9958f0a7ae1df715,content_0642dc7f62d4f780,2025-01-30,2.0,2,0.118238,1.881762
37,client_9958f0a7ae1df715,content_4bb2b9cacd16054d,2025-01-30,2.0,0,0.131544,1.868456
127,client_9958f0a7ae1df715,content_de7b08874af74c00,2025-01-30,0.0,2,1.488334,1.488334
115,client_9958f0a7ae1df715,content_d02be57d816cf3d7,2025-01-30,0.0,0,1.107518,1.107518
36,client_9958f0a7ae1df715,content_4b90d8f71a8d9c59,2025-01-30,1.0,2,0.081176,0.918824
23,client_9958f0a7ae1df715,content_3921fa1f7890fd63,2025-01-30,1.0,2,0.178917,0.821083
58,client_9958f0a7ae1df715,content_67c0464f614c0606,2025-01-30,1.0,0,0.267213,0.732787


In [37]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
0,gsc_impressions,0.759139
3,gsc_avg_position,0.227536
2,ctr,0.012069
1,gsc_clicks,0.001256
4,ga4_pageviews,0.000000
5,ga4_sessions,0.000000
6,ga4_engaged_sessions,0.000000
7,sessions_organic,0.000000
8,sessions_direct,0.000000
9,sessions_referral,0.000000


The largest observed prediction errors occur on content items where next-day clicks differ substantially from the model prediction. In this run, the model relies mainly on GSC impressions and average position, while the remaining features contribute much less. The evaluation covers only a short January window and uses next-day clicks as a proxy for review value, so the result should be treated as directional decision-support evidence rather than a validated refresh predictor.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.